# analysis_full.ipynb — Глава 3: Модели скорости движения для SAR-БПЛА
**Магистерская диссертация.** Полный анализ: EDA → Tobler → CatBoost-base → CatBoost-opt → LSTM HikingTTE+GIS.
Запуск: `poetry run jupyter notebook`


In [ ]:
import os, pickle, warnings
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings('ignore')
os.makedirs('figures', exist_ok=True)

plt.rcParams.update({
    'font.family': 'DejaVu Sans',
    'axes.unicode_minus': False,
    'figure.dpi': 100,
})

CACHE_DIR = 'cache'
MODEL_DIR = 'models'
FIG_DIR   = 'figures'
DPI       = 150

def savefig(name):
    plt.tight_layout()
    plt.savefig(f'{FIG_DIR}/{name}', dpi=DPI, bbox_inches='tight')
    plt.show()
    print(f'  -> сохранено: figures/{name}')

def tte_mape_compute(df_part):
    """TTE MAPE и MAE (в часах) по треку. Back 70% трека, min 20 сегментов."""
    res = []
    for tid, g in df_part.groupby('track_id'):
        g = g.reset_index(drop=True)
        n = len(g)
        if n < 20: continue
        sp = max(3, int(n * 0.3))
        back = g.iloc[sp:]
        if len(back) < 5: continue
        act  = (back['dist_m'] / 1000 / back['speed_kmh'].clip(0.3)).sum()
        pred = (back['dist_m'] / 1000 / back['pred'].clip(0.3)).sum()
        res.append((pred, act))
    if not res:
        return float('nan'), float('nan')
    r = np.array(res)
    mape  = (np.abs(r[:,0]-r[:,1]) / np.maximum(r[:,1], 1e-3)).mean() * 100
    mae_h = np.abs(r[:,0]-r[:,1]).mean()
    return mape, mae_h

print('Инициализация завершена.')


## Часть 1. EDA исходных данных hikr.org

Данные получены с hikr.org — базы данных GPX-треков альпийских маршрутов.
CSV-файл содержит мета-поля (длина, время, набор высоты, сложность SAC) и поле `gpx` с исходным GPX-XML.
Шаги обработки (ноутбук `02_build_segments`): парсинг GPX → расчёт сегментов → сегментные фильтры → расчёт уклона → Alpine bbox-фильтр.


In [ ]:
# 1.1 Загрузка мета-CSV (без GPX-поля)
CSV_PATH = 'gpx-tracks-from-hikr.org.csv'
META_COLS = ['_id','length_3d','user','start_time','max_elevation','uphill',
             'moving_time','end_time','max_speed','difficulty',
             'min_elevation','downhill','name','length_2d']

csv_df = pd.read_csv(CSV_PATH, usecols=META_COLS, low_memory=False)
csv_df = csv_df.dropna(subset=['_id'])  # убрать строки без id (продолжения GPX-XML)
csv_df = csv_df.drop_duplicates(subset=['_id'])

n_raw = len(csv_df)
n_valid = csv_df['moving_time'].notna().sum()
csv_df['speed_kmh_meta'] = (csv_df['length_2d'] / csv_df['moving_time'] * 3.6).clip(0, 10)
csv_df['length_km']      = (csv_df['length_2d'] / 1000).clip(0, 40)
csv_df['elev_gain_m']    = csv_df['uphill'].clip(0, 3000)

print(f'Треков в CSV:         {n_raw:,}')
print(f'  с valid moving_time: {n_valid:,} ({n_valid/n_raw*100:.1f}%)')

diff_raw = csv_df['difficulty'].value_counts().head(12)
print('\nСложность SAC (топ-10):')
print(diff_raw.to_string())


In [ ]:
# fig_01 - гистограммы мета-данных CSV
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

sub = csv_df.dropna(subset=['speed_kmh_meta','length_km','elev_gain_m'])
axes[0].hist(sub['speed_kmh_meta'].dropna(), bins=50, color='steelblue', edgecolor='none')
axes[0].set(xlabel='Скорость трека (км/ч)', ylabel='Количество треков',
            title='Средняя скорость (length_2d/moving_time)')

axes[1].hist(sub['length_km'].dropna(), bins=50, color='darkorange', edgecolor='none')
axes[1].set(xlabel='Длина трека (км)', ylabel='', title='Длина маршрута')

axes[2].hist(sub['elev_gain_m'].dropna(), bins=50, color='forestgreen', edgecolor='none')
axes[2].set(xlabel='Набор высоты (м)', ylabel='', title='Набор высоты (uphill)')

savefig('fig_01_hikr_raw_meta.png')

print('\n### Ключевые числа (Часть 1.1):')
print(f'  Треков:             {n_raw:,}')
print(f'  Медиана скорость:  {sub["speed_kmh_meta"].median():.2f} км/ч')
print(f'  Медиана длина:            {sub["length_km"].median():.1f} км')
print(f'  Медиана набор высоты:     {sub["elev_gain_m"].median():.0f} м')


In [ ]:
# 1.2 Воронка обработки данных - числа из финального датасета
df = pickle.load(open(f'{CACHE_DIR}/hikr_model_df.pkl', 'rb'))

alps_mask = ((df['lat_mid']>=43)&(df['lat_mid']<=49)&
             (df['lon_mid']>=5)&(df['lon_mid']<=20))
alps_df = df[alps_mask].reset_index(drop=True)

n_full_tracks = df['track_id'].nunique()
n_full_segs   = len(df)
n_alps_tracks = alps_df['track_id'].nunique()
n_alps_segs   = len(alps_df)
pct_alps = n_alps_tracks / n_full_tracks * 100

print('+----------------------------------------------------------------------+')
print('|           ВОРОНКА ПОДГОТОВКИ ДАННЫХ hikr.org                       |')
print('+------------------+--------------+---------------+------------------+')
print('| Этап             |   Треков     |  Сегментов    |   % треков       |')
print('+------------------+--------------+---------------+------------------+')
print(f'| 1. Сырой CSV     | {n_raw:>12,} |       -       |     100.0%       |')
print(f'| 2. После парсинга| {n_full_tracks:>12,} | {n_full_segs:>13,} |  {n_full_tracks/n_raw*100:>7.1f}%        |')
print(f'|    + сегм. фильт.|              |               |                  |')
print(f'| 3. Альпы (bbox)  | {n_alps_tracks:>12,} | {n_alps_segs:>13,} |  {pct_alps:>7.1f}%        |')
print('|    43-49N, 5-20E |              |               |                  |')
print('+------------------+--------------+---------------+------------------+')
print()
print(f'Финальный датасет: {n_alps_tracks:,} треков, {n_alps_segs:,} сегментов ({pct_alps:.1f}% от исходного)')
print(f'Сегментные фильтры (ноутбук 02): dist 1-1000 м, dt 1-3600 с, speed 0.3-15 км/ч')


## Часть 2. EDA финального датасета сегментов

Работаем с Alpine-подмножеством `hikr_model_df.pkl`: треки в bbox 43–49°N, 5–20°E.
Каждая строка — один сегмент трека (пара соседних GPS-точек).


In [ ]:
# 2.1 Базовая статистика
print(f'Сегментов: {len(alps_df):,}   треков: {alps_df["track_id"].nunique():,}')
print(f'Столбцы: {list(alps_df.columns)}')
print()
print(alps_df['speed_kmh'].describe().rename({
    'count':'count','mean':'среднее','std':'std',
    'min':'min','25%':'p25','50%':'медиана','75%':'p75','max':'max'
}).to_string())


In [ ]:
# !pip install contextily

In [ ]:
# fig_05 - Карта сегментов (5000 случайных точек)
# samp = alps_df.sample(5000, random_state=42)
# fig, ax = plt.subplots(figsize=(10, 6))
# sc = ax.scatter(samp['lon_mid'], samp['lat_mid'], c=samp['speed_kmh'],
#                 cmap='RdYlGn', s=2, alpha=0.6, vmin=0.5, vmax=6)
# plt.colorbar(sc, ax=ax, label='speed_kmh')

# # Alps bbox
# from matplotlib.patches import Rectangle
# ax.add_patch(Rectangle((5,43),15,6, linewidth=2, edgecolor='red',
#                          facecolor='none', label='Альпийский регион'))
# ax.set(xlabel='Долгота (degE)', ylabel='Широта (degN)',
#        title=f'Географическое покрытие данных (5000 сэмплов из {len(alps_df):,} сегментов)')
# ax.legend()
# savefig('fig_05_segments_geography.png')

# fig_05 - Карта сегментов (5000 случайных точек) с подложкой OSM
import contextily as ctx
from matplotlib.patches import Rectangle

samp = alps_df.sample(5000, random_state=42)

fig, ax = plt.subplots(figsize=(11, 7))

# scatter в географических координатах (EPSG:4326)
sc = ax.scatter(samp['lon_mid'], samp['lat_mid'], c=samp['speed_kmh'],
                cmap='RdYlGn', s=4, alpha=0.7, vmin=0.5, vmax=6,
                edgecolors='none', zorder=2)
plt.colorbar(sc, ax=ax, label='speed_kmh', shrink=0.7)

# Alpine bbox
# ax.add_patch(Rectangle((5, 43), 15, 6, linewidth=2, edgecolor='red',
#                        facecolor='none', label='Альпийский bbox (43-49N, 5-20E)',
#                        zorder=3))

# Подложка карты (OSM через contextily)
# crs='EPSG:4326' говорит contextily что данные в lat/lon, а не в Web Mercator
ctx.add_basemap(ax, crs='EPSG:4326', source=ctx.providers.OpenStreetMap.Mapnik,
                attribution_size=6, zorder=1)

ax.set(xlabel='Долгота (degE)', ylabel='Широта (degN)',
       title=f'Географическое покрытие данных hikr.org\n'
             f'({len(samp):,} сэмплов из {len(alps_df):,} сегментов)')
ax.legend(loc='upper right')
ax.set_xlim(samp['lon_mid'].min() - 1, samp['lon_mid'].max() + 1)
ax.set_ylim(samp['lat_mid'].min() - 0.5, samp['lat_mid'].max() + 0.5)

savefig('fig_05_segments_geography.png')

In [ ]:
# fig_06 - Распределение SAC difficulty
fig, ax = plt.subplots(figsize=(8, 4))
diff_counts = alps_df['difficulty'].value_counts().sort_index()
bars = ax.bar(diff_counts.index, diff_counts.values, color='steelblue', edgecolor='white')
for bar, val in zip(bars, diff_counts.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1000,
            f'{val:,}', ha='center', va='bottom', fontsize=9)
ax.set(xlabel='Сложность SAC (T1-T6)', ylabel='Количество сегментов',
       title='Распределение сложности SAC в финальном датасете',
       xticks=range(1,7),
       xticklabels=['T1 (лёгкая)','T2','T3','T4','T5','T6 (экстрем.)'])
savefig('fig_06_sac_distribution.png')

print('\n### Ключевые числа (сложность SAC):')
for k,v in diff_counts.items():
    print(f'  T{k}: {v:,} сегментов ({v/len(alps_df)*100:.1f}%)')


In [ ]:
# fig_07 - Распределение speed_kmh
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

spd = alps_df['speed_kmh']
ax1.hist(spd, bins=80, color='steelblue', edgecolor='none', density=True)
ax1.axvline(spd.median(), color='red', linestyle='--', label=f'Медиана {spd.median():.2f}')
ax1.axvline(spd.quantile(0.1), color='orange', linestyle=':', label=f'p10={spd.quantile(0.1):.2f}')
ax1.axvline(spd.quantile(0.9), color='orange', linestyle=':', label=f'p90={spd.quantile(0.9):.2f}')
ax1.set(xlabel='Скорость (км/ч)', ylabel='Плотность', title='Гистограмма скорости')
ax1.legend(fontsize=9)

ax2.boxplot(spd, vert=True, patch_artist=True,
            boxprops=dict(facecolor='steelblue', alpha=0.6))
ax2.set(ylabel='Скорость (км/ч)', title='Ящик с усами', xticks=[1], xticklabels=['speed_kmh'])

savefig('fig_07_speed_distribution.png')

print('\n### Ключевые числа (speed_kmh):')
print(f'  Медиана:  {spd.median():.3f} км/ч')
print(f'  p10-p90:  {spd.quantile(0.1):.3f} - {spd.quantile(0.9):.3f} км/ч')
print(f'  Std:      {spd.std():.3f} км/ч')
print(f'  Min/Max:  {spd.min():.3f} / {spd.max():.3f} км/ч')


In [ ]:
# fig_08 - Базовые признаки (2x2)
alps_df['signed_slope_deg_tmp'] = np.degrees(
    np.arctan2(alps_df['elev_diff_m'].values, np.maximum(alps_df['dist_m'].values, 0.1))
)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

configs = [
    ('slope_deg',           'Уклон slope_deg',           0, 45,   'steelblue'),
    ('signed_slope_deg_tmp','Отклонения уклона',            -40, 40,  'darkorange'),
    ('dist_m',              'Длина сегмента dist_m (м)',      0, 300,  'forestgreen'),
    ('elev_diff_m',         'Перепад высоты elev_diff_m (м)',-100, 100,'purple'),
]

for ax, (col, xlabel, lo, hi, clr) in zip(axes, configs):
    data = alps_df[col].clip(lo, hi)
    ax.hist(data, bins=60, color=clr, edgecolor='none', density=True)
    ax.set(xlabel=xlabel, ylabel='Плотность', title=xlabel)

savefig('fig_08_base_features.png')
alps_df.drop(columns=['signed_slope_deg_tmp'], inplace=True)


## Часть 3. GIS-разметка из OSM

OSM-метки (`hikr_osm_labels.pkl`) содержат расстояния от midpoint каждого сегмента
до ближайших OSM-объектов: тропы, дороги, водоёма.
Метки выровнены построчно с Alpine-подмножеством датасета.


In [ ]:
# 3.1 Загрузка и присоединение OSM-меток
osm = pickle.load(open(f'{CACHE_DIR}/hikr_osm_labels.pkl', 'rb'))
print(f'OSM-меток: {len(osm["dist_trail_m"]):,} (совпадает с Alps df: {len(osm["dist_trail_m"])==len(alps_df)})')

alps_df['dist_trail_m'] = np.array(osm['dist_trail_m'], dtype=np.float32)
alps_df['dist_road_m']  = np.array(osm['dist_road_m'],  dtype=np.float32)
alps_df['dist_water_m'] = np.array(osm['dist_water_m'], dtype=np.float32)
print('Добавлены столбцы: dist_trail_m, dist_road_m, dist_water_m')


In [ ]:
# # fig_10 - Распределение расстояний до OSM-объектов
# fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# for ax, (col, label, color) in zip(axes, [
#     ('dist_trail_m', 'До тропы (м)',  'steelblue'),
#     ('dist_road_m',  'До дороги (м)', 'darkorange'),
#     ('dist_water_m', 'До воды (м)',   'teal'),
# ]):
#     data = alps_df[col].clip(0, 5000)
#     ax.hist(data, bins=60, color=color, edgecolor='none', density=True)
#     ax.axvline(data.median(), color='red', linestyle='--',
#                label=f'Медиана {data.median():.0f} м')
#     ax.set(xlabel=label, ylabel='Плотность', title=f'Расстояние {label}')
#     ax.legend(fontsize=9)

# savefig('fig_10_osm_distances.png')

# print('\n### Ключевые числа (OSM-расстояния):')
# for col, name in [('dist_trail_m','тропа'),('dist_road_m','дорога'),('dist_water_m','вода')]:
#     d = alps_df[col]
#     print(f'  {name}: медиана={d.median():.0f} м, p75={d.quantile(0.75):.0f} м, >5км={( d>5000).mean()*100:.1f}%')

# fig_10 - Распределение расстояний до OSM-объектов (адаптивный клип под каждый класс)
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

configs = [
    ('dist_trail_m', 'До тропы (м)',  'steelblue',  100,  50),
    ('dist_road_m',  'До дороги (м)', 'darkorange', 3000, 60),
    ('dist_water_m', 'До воды (м)',   'teal',       1000, 60),
]

for ax, (col, label, color, clip_max, bins) in zip(axes, configs):
    raw = alps_df[col]
    data = raw.clip(0, clip_max)
    
    ax.hist(data, bins=bins, color=color, edgecolor='none', density=True)
    
    median = raw.median()
    p75 = raw.quantile(0.75)
    pct_over_clip = (raw > clip_max).mean() * 100
    
    ax.axvline(median, color='red', linestyle='--', lw=1.5,
               label=f'Медиана {median:.0f} м')
    ax.axvline(p75, color='black', linestyle=':', lw=1.2,
               label=f'p75 = {p75:.0f} м')
    
    ax.set(xlabel=label, ylabel='Плотность', title=f'Расстояние {label}')
    ax.set_xlim(0, clip_max)
    
    # подпись о том, какая доля данных за пределами окна
    # if pct_over_clip > 0.5:
    #     ax.text(0.98, 0.70, f'>{clip_max} м: {pct_over_clip:.1f}%',
    #             transform=ax.transAxes, ha='right', fontsize=9,
    #             bbox=dict(facecolor='white', edgecolor='gray', alpha=0.85))
    
    ax.legend(fontsize=9, loc='upper right')

plt.tight_layout()
savefig('fig_10_osm_distances.png')

print('\n### Ключевые числа (OSM-расстояния):')
for col, name in [('dist_trail_m','тропа'), ('dist_road_m','дорога'), ('dist_water_m','вода')]:
    d = alps_df[col]
    print(f'  {name}: медиана={d.median():.0f} м, p75={d.quantile(0.75):.0f} м, '
          f'p90={d.quantile(0.9):.0f} м, p99={d.quantile(0.99):.0f} м, '
          f'>5км={(d>5000).mean()*100:.1f}%')

In [ ]:
# 3.3 Бинарные признаки
alps_df['on_trail']   = (alps_df['dist_trail_m'] < 50).astype(int)
alps_df['on_road']    = (alps_df['dist_road_m']  < 30).astype(int)
alps_df['near_water'] = (alps_df['dist_water_m'] < 100).astype(int)
alps_df['off_trail']  = ((alps_df['on_trail']==0) & (alps_df['on_road']==0)).astype(int)

# fig_11 - доли бинарных признаков
features_binary = {
    'on_trail':   'На тропе (< 50 м)',
    'on_road':    'На дороге (< 30 м)',
    'near_water': 'Близко к воде (< 100 м)',
    'off_trail':  'Бездорожье',
}
shares = {v: alps_df[k].mean()*100 for k,v in features_binary.items()}

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(list(shares.keys()), list(shares.values()), color='steelblue')
for bar, val in zip(bars, shares.values()):
    ax.text(val+0.3, bar.get_y()+bar.get_height()/2, f'{val:.1f}%',
            va='center', fontsize=10)
ax.set(xlabel='Доля сегментов (%)', title='Доли бинарных OSM-признаков', xlim=(0,105))
savefig('fig_11_osm_binary_shares.png')

print('\n### Ключевые числа (OSM-бинарные):')
for k,v in shares.items():
    print(f'  {k}: {v:.1f}%')


In [ ]:
# 3.4 Непрерывные log-признаки и signed_slope_deg
alps_df['log_dist_trail']    = np.log1p(alps_df['dist_trail_m'].clip(0, 5000))
alps_df['log_dist_road']     = np.log1p(alps_df['dist_road_m'].clip(0, 5000))
alps_df['signed_slope_deg']  = np.degrees(
    np.arctan2(alps_df['elev_diff_m'].values,
               np.maximum(alps_df['dist_m'].values, 0.1))
)

# 3.5 custom_difficulty
slope = alps_df['slope_deg'].values
road  = alps_df['on_road'].values.astype(bool)
trail = alps_df['on_trail'].values.astype(bool)
water = alps_df['near_water'].values.astype(bool)

base = np.where(road,                  1,
       np.where(trail & (slope < 15),  2,
       np.where(trail | (slope < 25),  3, 4)))
alps_df['custom_difficulty'] = np.where(water & ~road,
                                         np.minimum(base+1, 5),
                                         base).astype(int)
print('Добавлены фичи: log_dist_trail, log_dist_road, signed_slope_deg, custom_difficulty')
print(alps_df['custom_difficulty'].value_counts().sort_index().rename('segments').to_string())


In [ ]:
# fig_12 - Crosstab SAC vs custom_difficulty
import matplotlib.colors as mcolors

ct = pd.crosstab(alps_df['difficulty'], alps_df['custom_difficulty'],
                  rownames=['SAC (T1-T6)'], colnames=['custom_difficulty (1-5)'],
                  normalize='index') * 100

fig, ax = plt.subplots(figsize=(8, 5))
im = ax.imshow(ct.values, aspect='auto', cmap='YlOrRd', vmin=0, vmax=100)
plt.colorbar(im, ax=ax, label='% сегментов SAC vs custom')
ax.set(xticks=range(len(ct.columns)), xticklabels=[f'C{c}' for c in ct.columns],
       yticks=range(len(ct.index)),   yticklabels=[f'T{r}' for r in ct.index],
       xlabel='custom_difficulty', ylabel='SAC T1-T6',
       title='Соответствие SAC vs custom_difficulty')

for i in range(len(ct.index)):
    for j in range(len(ct.columns)):
        val = ct.values[i,j]
        ax.text(j, i, f'{val:.0f}', ha='center', va='center',
                fontsize=9, color='black' if val < 60 else 'white')

savefig('fig_12_sac_vs_custom.png')

# fig_13 - гистограмма custom_difficulty
fig, ax = plt.subplots(figsize=(7, 4))
cd_cnt = alps_df['custom_difficulty'].value_counts().sort_index()
bars = ax.bar(cd_cnt.index, cd_cnt.values, color='steelblue', edgecolor='white')
for bar, v in zip(bars, cd_cnt.values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+500,
            f'{v:,}', ha='center', va='bottom', fontsize=9)
ax.set(xlabel='custom_difficulty (1=дорога, 5=сложное+вода)',
       ylabel='Сегментов', title='Распределение кастомной сложности')
savefig('fig_13_custom_distribution.png')


In [ ]:
# 3.6 Таблица признаков для итоговой модели
print('\n### Таблица признаков (CatBoost-opt / LSTM):')
print(f'{"Фича":<22} {"Тип":<12} {"Описание"}')
print('-'*75)
feats = [
    ('signed_slope_deg',  'float',    'Уклон со знаком (+ вверх, - вниз), deg'),
    ('elev_diff_m',       'float',    'Перепад высот сегмента, м'),
    ('on_trail',          'cat 0/1',  'Сегмент на OSM-тропе (расстояние < 50 м)'),
    ('on_road',           'cat 0/1',  'Сегмент на OSM-дороге (расстояние < 30 м)'),
    ('off_trail',         'cat 0/1',  'Ни тропа, ни дорога'),
    ('near_water',        'cat 0/1',  'Ближе 100 м к водоёму (OSM)'),
    ('log_dist_trail',    'float',    'log(расстояние до тропы + 1)'),
    ('log_dist_road',     'float',    'log(расстояние до дороги + 1)'),
    ('custom_difficulty', 'cat 1-5',  'Кастомная сложность из OSM (1=дорога, 5=сложн.+вода)'),
]
for name, typ, desc in feats:
    print(f'{name:<22} {typ:<12} {desc}')


## Часть 4. Canonical split — train/val/test по track_id

Единый train/val/test split для всех моделей: 70% / 15% / 15% по треку.
Разбиение по `track_id` — сегменты одного трека не могут быть в разных частях.
Seed=42, `np.random.default_rng(42)`.


In [ ]:
# 4.1 Загрузка split
sp = pickle.load(open(f'{CACHE_DIR}/canonical_split.pkl', 'rb'))
tr_ids, val_ids, te_ids = sp['tr_ids'], sp['val_ids'], sp['te_ids']

tr_df  = alps_df[alps_df['track_id'].isin(tr_ids)].copy()
val_df = alps_df[alps_df['track_id'].isin(val_ids)].copy()
te_df  = alps_df[alps_df['track_id'].isin(te_ids)].copy()

print("{:<10} {:>8} {:>12} {:>10} {:>12}".format(
    'Часть', 'Треков', 'Сегментов', '% треков', '% сегментов'))
print('-'*52)
total_t = len(tr_ids) + len(val_ids) + len(te_ids)
total_s = len(alps_df)
for name, ids, part_df in [('Train', tr_ids, tr_df), ('Val', val_ids, val_df), ('Test', te_ids, te_df)]:
    print(f'{name:<10} {len(ids):>8,} {len(part_df):>12,} {len(ids)/total_t*100:>9.1f}% {len(part_df)/total_s*100:>11.1f}%')
print('-'*52)
print("{:<10} {:>8,} {:>12,}".format('Итого', total_t, total_s))

# Sanity check
assert len(tr_ids & val_ids) == 0
assert len(tr_ids & te_ids)  == 0
assert len(val_ids & te_ids) == 0
print('\nSanity check: пересечений между частями нет ok')


In [ ]:
# fig_14 - распределение speed_kmh в train/val/test
fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
for ax, (name, part_df, color) in zip(axes, [
    ('Train', tr_df, 'steelblue'),
    ('Val',   val_df,'darkorange'),
    ('Test',  te_df, 'forestgreen'),
]):
    ax.hist(part_df['speed_kmh'], bins=60, color=color, edgecolor='none', density=True, alpha=0.8)
    ax.axvline(part_df['speed_kmh'].median(), color='red', linestyle='--',
               label=f'Медиана {part_df["speed_kmh"].median():.2f}')
    ax.set(xlabel='Скорость (км/ч)', ylabel='Плотность' if ax==axes[0] else '',
           title=f'{name} ({len(part_df):,} сегм.)')
    ax.legend(fontsize=9)

savefig('fig_14_split_distributions.png')

print('\n### Ключевые числа (split):')
for name, part_df in [('Train',tr_df),('Val',val_df),('Test',te_df)]:
    spd = part_df['speed_kmh']
    print(f'  {name}: медиана={spd.median():.3f}, std={spd.std():.3f}')


## Часть 5. Baseline — формула Тоблера

$$v_{\text{Tobler}}(s) = 6 \cdot \exp\left(-3.5 \left|\tan(s) + 0.05\right|\right)$$

где $s$ — уклон в радианах. Физическая модель без обучения.


In [ ]:
# 5.1 Tobler baseline
def tobler_kmh(slope_deg):
    s = np.asarray(slope_deg, dtype=float)
    return 6 * np.exp(-3.5 * np.abs(np.tan(np.radians(s)) + 0.05))

te_df = te_df.copy()
te_df['pred'] = tobler_kmh(te_df['slope_deg'])

mae_tobler  = mean_absolute_error(te_df['speed_kmh'], te_df['pred'])
rmse_tobler = mean_squared_error(te_df['speed_kmh'],  te_df['pred'])**0.5
r2_tobler   = r2_score(te_df['speed_kmh'],            te_df['pred'])
mape_seg    = (np.abs(te_df['pred']-te_df['speed_kmh'])/
               te_df['speed_kmh'].clip(0.1)).mean()*100

tte_mape_tobler, tte_mae_tobler = tte_mape_compute(te_df)

print('=== Baseline Tobler ===')
print(f'  MAE  (сегм.):  {mae_tobler:.4f} км/ч')
print(f'  RMSE (сегм.):  {rmse_tobler:.4f} км/ч')
print(f'  R^2:            {r2_tobler:.4f}')
print(f'  MAPE (сегм.):  {mape_seg:.2f}%')
print(f'  TTE MAPE:      {tte_mape_tobler:.2f}%')
print(f'  TTE MAE:       {tte_mae_tobler:.4f} ч')
print()
print('### Ключевые числа (Tobler):')
print(f'  TTE MAPE = {tte_mape_tobler:.2f}%  |  per-сегмент MAE = {mae_tobler:.4f} км/ч')


In [ ]:
# fig_15 - Predicted vs Actual scatter (Tobler)
samp = te_df.sample(min(5000, len(te_df)), random_state=42)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(samp['speed_kmh'], samp['pred'], alpha=0.15, s=6, color='steelblue')
lim = [0, 11]
ax.plot(lim, lim, 'r--', lw=1.5, label='y = x')
ax.set(xlabel='Фактическая скорость (км/ч)', ylabel='Предсказанная скорость (км/ч)',
       title=f'Tobler: predicted vs actual\nMAE={mae_tobler:.3f}, R^2={r2_tobler:.3f}',
       xlim=lim, ylim=lim)
ax.legend()
savefig('fig_15_tobler_scatter.png')


In [ ]:
# fig_16 - Аналитическая кривая Tobler
slopes_range = np.linspace(-40, 40, 400)
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(slopes_range, tobler_kmh(slopes_range), 'b-', lw=2.5)
ax.axvline(0,  color='gray', linestyle=':')
ax.axhline(6,  color='gray', linestyle=':', label='Максимум ~6 км/ч')
opt_slope = slopes_range[np.argmax(tobler_kmh(slopes_range))]
ax.axvline(opt_slope, color='orange', linestyle='--',
           label=f'Оптимум: {opt_slope:.1f}deg')
ax.set(xlabel='Уклон (deg)', ylabel='Скорость (км/ч)',
       title='Функция Тоблера: скорость vs уклон')
ax.legend()
savefig('fig_16_tobler_curve.png')


## Часть 6. CatBoost-base (SAC difficulty, 7 фичей)

Модель обучена на полном наборе hikr.org треков с признаками:
`log_dist, slope_deg, slope_sq, elev_diff_m, going_up, slope_x_up, difficulty (SAC T1-T6)`.


In [ ]:
from catboost import CatBoostRegressor, Pool

CB_BASE_FEATS = ['log_dist','slope_deg','slope_sq','elev_diff_m',
                 'going_up','slope_x_up','difficulty']
CB_BASE_CAT   = ['going_up','difficulty']

cb_base = CatBoostRegressor(verbose=0)
cb_base.load_model(f'{MODEL_DIR}/travel_time_hikr.cbm')

# Проверяем наличие нужных колонок (должны быть в df из NB-03)
for col in CB_BASE_FEATS:
    if col not in te_df.columns:
        print(f'  Вычисляем {col}...')
        if col == 'log_dist':   te_df[col] = np.log1p(te_df['dist_m'])
        if col == 'slope_sq':   te_df[col] = te_df['slope_deg']**2
        if col == 'going_up':   te_df[col] = (te_df['elev_diff_m']>0).astype(int)
        if col == 'slope_x_up': te_df[col] = te_df['slope_deg'] * te_df.get('going_up', 0)

X_te_b = te_df[CB_BASE_FEATS]
te_df['pred'] = cb_base.predict(Pool(X_te_b, cat_features=CB_BASE_CAT))

mae_b  = mean_absolute_error(te_df['speed_kmh'], te_df['pred'])
rmse_b = mean_squared_error(te_df['speed_kmh'],  te_df['pred'])**0.5
r2_b   = r2_score(te_df['speed_kmh'],            te_df['pred'])
tte_mape_b, tte_mae_b = tte_mape_compute(te_df)

# Train metrics
for col in CB_BASE_FEATS:
    if col not in tr_df.columns:
        if col == 'log_dist':   tr_df[col] = np.log1p(tr_df['dist_m'])
        if col == 'slope_sq':   tr_df[col] = tr_df['slope_deg']**2
        if col == 'going_up':   tr_df[col] = (tr_df['elev_diff_m']>0).astype(int)
        if col == 'slope_x_up': tr_df[col] = tr_df['slope_deg'] * tr_df.get('going_up',0)
p_tr = cb_base.predict(Pool(tr_df[CB_BASE_FEATS], cat_features=CB_BASE_CAT))
mae_b_tr = mean_absolute_error(tr_df['speed_kmh'], p_tr)

print('=== CatBoost-base (SAC, 7 фичей) ===')
print(f'  Train MAE:     {mae_b_tr:.4f} км/ч')
print(f'  Test MAE:      {mae_b:.4f} км/ч')
print(f'  Test RMSE:     {rmse_b:.4f} км/ч')
print(f'  Test R^2:       {r2_b:.4f}')
print(f'  TTE MAPE:      {tte_mape_b:.2f}%')
print(f'  TTE MAE:       {tte_mae_b:.4f} ч')
print()
print('### Ключевые числа (CB-base):')
print(f'  TTE MAPE = {tte_mape_b:.2f}%  |  Test MAE = {mae_b:.4f} км/ч  |  R^2 = {r2_b:.4f}')


In [ ]:
# fig_17 - Feature Importance CatBoost-base
fi_b = pd.Series(cb_base.get_feature_importance(),
                  index=CB_BASE_FEATS).sort_values(ascending=True)
fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.barh(fi_b.index, fi_b.values, color='steelblue')
for bar, v in zip(bars, fi_b.values):
    ax.text(v+0.2, bar.get_y()+bar.get_height()/2,
            f'{v:.1f}', va='center', fontsize=9)
ax.set(xlabel='Feature Importance (%)', title='CatBoost-base: важность признаков (7 фичей)',
       xlim=(0, fi_b.max()+5))
savefig('fig_17_cb_base_fi.png')


## Часть 7. CatBoost-opt (кастомная OSM-сложность + Optuna, 9 фичей)

Модель обучена с GIS-признаками из OSM + тюнингом гиперпараметров Optuna (40 триалов, TPE-сэмплер).
Признаки: `signed_slope_deg, elev_diff_m, on_trail, on_road, off_trail, near_water, log_dist_trail, log_dist_road, custom_difficulty`.


In [ ]:
CB_OPT_FEATS = ['signed_slope_deg','elev_diff_m','on_trail','on_road',
                'off_trail','near_water','log_dist_trail','log_dist_road','custom_difficulty']
CB_OPT_CAT   = ['on_trail','on_road','off_trail','near_water','custom_difficulty']

cb_opt = CatBoostRegressor(verbose=0)
cb_opt.load_model(f'{MODEL_DIR}/travel_time_opt.cbm')

X_te_o = te_df[CB_OPT_FEATS]
te_df['pred'] = cb_opt.predict(Pool(X_te_o, cat_features=CB_OPT_CAT))

mae_o  = mean_absolute_error(te_df['speed_kmh'], te_df['pred'])
rmse_o = mean_squared_error(te_df['speed_kmh'],  te_df['pred'])**0.5
r2_o   = r2_score(te_df['speed_kmh'],            te_df['pred'])
tte_mape_o, tte_mae_o = tte_mape_compute(te_df)

p_tr_o = cb_opt.predict(Pool(tr_df[CB_OPT_FEATS], cat_features=CB_OPT_CAT))
mae_o_tr = mean_absolute_error(tr_df['speed_kmh'], p_tr_o)

print('=== CatBoost-opt (GIS+Optuna, 9 фичей) ===')
print(f'  Train MAE:     {mae_o_tr:.4f} км/ч')
print(f'  Test MAE:      {mae_o:.4f} км/ч')
print(f'  Test RMSE:     {rmse_o:.4f} км/ч')
print(f'  Test R^2:       {r2_o:.4f}')
print(f'  TTE MAPE:      {tte_mape_o:.2f}%')
print(f'  TTE MAE:       {tte_mae_o:.4f} ч')
print()
print('### Ключевые числа (CB-opt):')
print(f'  TTE MAPE = {tte_mape_o:.2f}%  |  Test MAE = {mae_o:.4f} км/ч  |  R^2 = {r2_o:.4f}')


In [ ]:
# fig_18 - FI CatBoost-opt vs base (side-by-side)
fi_o = pd.Series(cb_opt.get_feature_importance(), index=CB_OPT_FEATS).sort_values(ascending=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, (fi, title, color) in zip(axes, [
    (fi_b, 'CatBoost-base (7 фичей)', 'steelblue'),
    (fi_o, 'CatBoost-opt (9 фичей)',  'darkorange'),
]):
    bars = ax.barh(fi.index, fi.values, color=color)
    for bar, v in zip(bars, fi.values):
        ax.text(v+0.2, bar.get_y()+bar.get_height()/2,
                f'{v:.1f}', va='center', fontsize=9)
    ax.set(xlabel='Feature Importance (%)', title=title, xlim=(0, fi.max()+6))

savefig('fig_18_cb_opt_fi.png')


In [ ]:
# fig_19/20 - SHAP анализ на 2000 тест-сегментах
try:
    import shap
    samp_shap = te_df.sample(2000, random_state=42)
    X_shap = samp_shap[CB_OPT_FEATS]
    pool_shap = Pool(X_shap, cat_features=CB_OPT_CAT)
    explainer = shap.TreeExplainer(cb_opt)
    shap_vals = explainer.shap_values(X_shap)

    # fig_19 - beeswarm
    fig, ax = plt.subplots(figsize=(9, 6))
    shap.summary_plot(shap_vals, X_shap, feature_names=CB_OPT_FEATS,
                      show=False, max_display=9)
    plt.title('SHAP Beeswarm - CatBoost-opt (2000 test сегментов)')
    savefig('fig_19_shap_beeswarm.png')

    # fig_20 - mean |SHAP|
    mean_abs_shap = pd.Series(np.abs(shap_vals).mean(axis=0),
                               index=CB_OPT_FEATS).sort_values(ascending=True)
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.barh(mean_abs_shap.index, mean_abs_shap.values, color='darkorange')
    for bar, v in zip(bars, mean_abs_shap.values):
        ax.text(v+0.002, bar.get_y()+bar.get_height()/2,
                f'{v:.3f}', va='center', fontsize=9)
    ax.set(xlabel='mean |SHAP value|', title='SHAP Mean |value| - CatBoost-opt')
    savefig('fig_20_shap_meanabs.png')

    print('SHAP анализ выполнен.')
except ImportError:
    print('shap не установлен - пропускаем SHAP-анализ')
except Exception as e:
    print(f'SHAP ошибка: {e}')


In [ ]:
# fig_21 - Кривые скорость-уклон для 4 сценариев
slopes_range = np.linspace(-40, 40, 200)

def make_scenario(model, slopes, on_trail=0, on_road=0, near_water=0):
    n = len(slopes)
    off_trail_v = int((on_trail == 0) and (on_road == 0))
    dt = np.full(n, 50.0 if on_trail else 1000.0)
    dr = np.full(n, 30.0 if on_road  else 1000.0)
    slope_abs = np.abs(slopes)
    trail = bool(on_trail); road = bool(on_road); wtr = bool(near_water)
    base = np.where(road, 1,
           np.where(trail & (slope_abs < 15), 2,
           np.where(trail | (slope_abs < 25), 3, 4)))
    cd = np.where(wtr and not road, np.minimum(base+1, 5), base).astype(int)
    elev = np.tan(np.radians(slope_abs)) * 100 * np.sign(slopes)
    X = pd.DataFrame({
        'signed_slope_deg': slopes,
        'elev_diff_m':      elev,
        'on_trail':         on_trail,
        'on_road':          on_road,
        'off_trail':        off_trail_v,
        'near_water':       near_water,
        'log_dist_trail':   np.log1p(dt.clip(0,5000)),
        'log_dist_road':    np.log1p(dr.clip(0,5000)),
        'custom_difficulty': cd,
    })
    return model.predict(Pool(X, cat_features=CB_OPT_CAT))

scenarios = [
    ('Бездорожье',      dict(on_trail=0, on_road=0, near_water=0), '#d62728'),
    ('Тропа',           dict(on_trail=1, on_road=0, near_water=0), '#2ca02c'),
    ('Дорога',          dict(on_trail=0, on_road=1, near_water=0), '#1f77b4'),
    ('Бездорожье+вода', dict(on_trail=0, on_road=0, near_water=1), '#9467bd'),
]

fig, ax = plt.subplots(figsize=(10, 5))
for label, kw, color in scenarios:
    spds = make_scenario(cb_opt, slopes_range, **kw)
    ax.plot(slopes_range, spds, color=color, lw=2, label=label)

ax.plot(slopes_range, tobler_kmh(slopes_range), 'k--', lw=1.5, alpha=0.6, label='Tobler (baseline)')
ax.set(xlabel='Уклон (deg)', ylabel='Скорость (км/ч)',
       title='Кривые скорость-уклон для 4 сценариев (CatBoost-opt)')
ax.legend(fontsize=10)
ax.grid(True, alpha=0.3)
savefig('fig_21_speed_slope_curves.png')


In [ ]:
# fig_22 - PDP (2x2) на 5000 тест-сегментах
pdp_feats = ['signed_slope_deg', 'log_dist_trail', 'log_dist_road', 'custom_difficulty']
pdp_sample = te_df.sample(min(5000, len(te_df)), random_state=42)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))
axes = axes.ravel()

pdp_ranges = {
    'signed_slope_deg': np.linspace(-35, 35, 50),
    'log_dist_trail':   np.linspace(0, 8.5, 40),
    'log_dist_road':    np.linspace(0, 8.5, 40),
    'custom_difficulty': np.arange(1, 6),
}

for ax, feat in zip(axes, pdp_feats):
    grid = pdp_ranges[feat]
    mean_preds = []
    X_base = pdp_sample[CB_OPT_FEATS].copy()
    for val in grid:
        X_tmp = X_base.copy()
        X_tmp[feat] = val
        preds = cb_opt.predict(Pool(X_tmp, cat_features=CB_OPT_CAT))
        mean_preds.append(preds.mean())

    ax.plot(grid, mean_preds, 'b-', lw=2)
    ax.set(xlabel=feat, ylabel='Средняя предсказанная скорость (км/ч)',
           title=f'PDP: {feat}')
    ax.grid(True, alpha=0.3)

savefig('fig_22_pdp.png')


In [ ]:
# fig_23 - Counterfactual эффект тропы/дороги
off_mask = (te_df['off_trail'] == 1)
cf_df = te_df[off_mask].copy()
print(f'Off-trail сегментов в test: {len(cf_df):,}')

if len(cf_df) > 0:
    # Базовый прогноз (as-is)
    pred_base = cb_opt.predict(Pool(cf_df[CB_OPT_FEATS], cat_features=CB_OPT_CAT))

    # Counterfactual: добавляем тропу
    cf_trail = cf_df[CB_OPT_FEATS].copy()
    cf_trail['on_trail']    = 1
    cf_trail['off_trail']   = 0
    cf_trail['log_dist_trail'] = np.log1p(20)  # ~20 м до тропы
    slope_t = cf_df['slope_deg'].values
    base_t  = np.where(slope_t < 15, 2, 3)
    cf_trail['custom_difficulty'] = np.where(
        cf_df['near_water'].values.astype(bool),
        np.minimum(base_t+1, 5), base_t)
    pred_trail = cb_opt.predict(Pool(cf_trail, cat_features=CB_OPT_CAT))

    # Counterfactual: добавляем дорогу
    cf_road = cf_df[CB_OPT_FEATS].copy()
    cf_road['on_road']   = 1
    cf_road['off_trail'] = 0
    cf_road['log_dist_road'] = np.log1p(10)  # ~10 м до дороги
    cf_road['custom_difficulty'] = 1
    pred_road = cb_opt.predict(Pool(cf_road, cat_features=CB_OPT_CAT))

    delta_trail = pred_trail - pred_base
    delta_road  = pred_road  - pred_base

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.boxplot([delta_trail, delta_road], labels=['delta тропа', 'delta дорога'],
               patch_artist=True,
               boxprops=dict(facecolor='steelblue', alpha=0.6),
               medianprops=dict(color='red', lw=2))
    ax.axhline(0, color='gray', linestyle='--')
    ax.set(ylabel='Прирост скорости (км/ч)',
           title='Контрфактуальный эффект: as-is vs тропа/дорога (off-trail сегменты)')
    savefig('fig_23_trail_road_effect.png')

    print('\n### Ключевые числа (counterfactual):')
    print(f'  delta тропа:  медиана={np.median(delta_trail):+.3f}, p25={np.percentile(delta_trail,25):+.3f}, p75={np.percentile(delta_trail,75):+.3f} км/ч')
    print(f'  delta дорога: медиана={np.median(delta_road):+.3f}, p25={np.percentile(delta_road,25):+.3f}, p75={np.percentile(delta_road,75):+.3f} км/ч')


In [ ]:
# 7.8 Гиперпараметры base vs opt
print('### Гиперпараметры моделей:')
print("{:<25} {:>15} {:>18}".format('Параметр', 'CB-base', 'CB-opt (Optuna)'))
print('-'*60)
params_opt = cb_opt.get_all_params()
params_base = cb_base.get_all_params()
for pname, base_val, desc in [
    ('depth',             params_base.get('depth','-'),        'Глубина дерева'),
    ('learning_rate',     round(params_base.get('learning_rate',float('nan')),4), 'Learning rate'),
    ('l2_leaf_reg',       round(params_base.get('l2_leaf_reg',float('nan')),3),   'L2-регуляризация'),
    ('iterations',        params_base.get('iterations','-'),    'Итераций'),
    ('bagging_temperature',params_base.get('bagging_temperature','-'), 'Bagging temp.'),
]:
    opt_val = params_opt.get(pname, '-')
    if isinstance(opt_val, float): opt_val = round(opt_val, 4)
    print("{:<25} {:>15} {:>18}".format(desc, str(base_val), str(opt_val)))


## Часть 8. LSTM HikingTTE + GIS

Рекуррентная нейросеть на основе архитектуры HikingTTE (Asako et al., 2022).
Расширена GIS-признаками: `on_trail, on_road, near_water, log_dist_trail, log_dist_road, custom_difficulty`.
Local head → время сегмента → скорость. Entire head → TTE всего маршрута.


In [ ]:
# 8.1 Загрузка LSTM чекпоинта
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F

    ckpt = torch.load(f'{MODEL_DIR}/hiking_tte_gis.pt', map_location='cpu', weights_only=False)
    print('Ключи чекпоинта:', list(ckpt.keys()))
    print()

    tm = ckpt['test_metrics']
    print('=== LSTM HikingTTE+GIS test_metrics ===')
    print(f'  TTE MAPE (entire path):  {tm.get("MAPE_entire", float("nan")):.2f}%')
    print(f'  TTE MAE (entire path):   {tm.get("MAE_entire_h", float("nan")):.4f} ч')
    print(f'  Per-segment MAPE:        {tm.get("MAPE_local", float("nan")):.2f}%')
    print()
    print('model_config:', ckpt['model_config'])
    print()
    lstm_ok = True

    h = ckpt.get('history', [])
    print(f'История обучения: {len(h)} записей', '(не сохранена)' if not h else '')

    print('\n### Ключевые числа (LSTM):')
    print(f'  TTE MAPE = {tm.get("MAPE_entire",float("nan")):.2f}%')
    print(f'  MAE_entire = {tm.get("MAE_entire_h",float("nan")):.4f} ч')
    print(f'  Vs. HikingTTE Asako (paper): 10.48% -> наша модель {tm.get("MAPE_entire",float("nan")):.2f}%')

except ImportError:
    print('torch не установлен - используем числа из сохранённых метрик')
    lstm_ok = False
    tm = {'MAPE_entire': 10.89, 'MAE_entire_h': 0.3956, 'MAPE_local': 26.25}
    print(f'  TTE MAPE = {tm["MAPE_entire"]:.2f}%  MAE = {tm["MAE_entire_h"]:.4f} ч')
except Exception as e:
    print(f'Ошибка загрузки LSTM: {e}')
    lstm_ok = False
    tm = {'MAPE_entire': 10.89, 'MAE_entire_h': 0.3956, 'MAPE_local': 26.25}


In [ ]:
# 8.2 История обучения (если есть)
try:
    if lstm_ok and len(ckpt.get('history',[])) > 0:
        hist = pd.DataFrame(ckpt['history'])
        fig, ax = plt.subplots(figsize=(9, 4))
        x_ax = hist['epoch'] if 'epoch' in hist.columns else hist.index
        if 'train_loss' in hist.columns:
            ax.plot(x_ax, hist['train_loss'], label='Train loss')
        if 'val_loss' in hist.columns:
            ax.plot(x_ax, hist['val_loss'],   label='Val loss')
        ax.set(xlabel='Эпоха', ylabel='Loss', title='Кривые обучения LSTM')
        ax.legend()
        savefig('fig_24_lstm_training.png')
    else:
        print('История обучения не сохранена в чекпоинте - график пропущен.')
        print('(Кривые обучения сохранены в hiking_tte_gis_training.png из ноутбука 06)')
except Exception as e:
    print(f'Пропуск истории обучения: {e}')


In [ ]:
# 8.3 Инференс LSTM на 4 test-треках
try:
    if not lstm_ok:
        raise RuntimeError('LSTM не загружен')

    class HikingTTEGIS(nn.Module):
        def __init__(self, n_point, n_attr, loc_dim=32, hidden=128, n_layers=2, dropout=0.2):
            super().__init__()
            self.loc_proj   = nn.Sequential(nn.Linear(n_point, loc_dim), nn.Tanh())
            self.lstm       = nn.LSTM(loc_dim+n_attr, hidden, n_layers, batch_first=True,
                                      dropout=dropout if n_layers>1 else 0.)
            self.local_head = nn.Sequential(
                nn.Linear(hidden,64), nn.LeakyReLU(),
                nn.Linear(64,32),    nn.LeakyReLU(),
                nn.Linear(32,1))
            self.attn_q      = nn.Linear(n_attr, hidden)
            self.entire_head = nn.Sequential(
                nn.Linear(hidden+n_attr,hidden), nn.LeakyReLU(),
                nn.Linear(hidden,hidden//2),     nn.LeakyReLU(),
                nn.Linear(hidden//2,1))

        def forward(self, X_seq, attr, mask):
            B,T,_ = X_seq.shape
            h,_ = self.lstm(torch.cat([self.loc_proj(X_seq),
                                        attr.unsqueeze(1).expand(-1,T,-1)],dim=-1))
            y_loc = F.softplus(self.local_head(h).squeeze(-1))
            q   = self.attn_q(attr).unsqueeze(-1)
            w   = torch.softmax(torch.bmm(h,q).squeeze(-1).masked_fill(mask==0,-1e9),dim=1)
            ctx = (h * w.unsqueeze(-1)).sum(1)
            y_ent = F.softplus(self.entire_head(torch.cat([ctx,attr],dim=-1)).squeeze(-1))
            return y_ent, y_loc, w

    model = HikingTTEGIS(**ckpt['model_config'])
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()

    x_mean = ckpt['x_scaler_mean'].astype(np.float32)
    x_std  = np.maximum(ckpt['x_scaler_std'].astype(np.float32), 1e-8)
    a_mean = ckpt['a_scaler_mean'].astype(np.float32)
    a_std  = np.maximum(ckpt['a_scaler_std'].astype(np.float32), 1e-8)

    POINT_FEATS = list(ckpt.get('point_features', [
        'signed_slope_deg','elev_diff_m','dist_m','on_trail','on_road',
        'off_trail','near_water','log_dist_trail','log_dist_road','custom_difficulty']))

    def _campbell(slope_deg):
        s = np.asarray(slope_deg,dtype=float)
        a,b,c,d,e = -1.4579,22.0787,76.3271,0.0525,3.2002e-4
        return np.maximum(c/(np.pi*b*(1+((s+a)/b)**2))+d+e*s, 0.01)*3.6

    ABILITY_SLOPES = np.array([-30,-20,-10,0,10,20,30],dtype=float)

    def default_attr(dist_m=100.):
        vhats = _campbell(ABILITY_SLOPES)
        return np.array([0.,0.,0.,0.,dist_m,*vhats],dtype=np.float32)

    def infer_track(gdf):
        """Инференс LSTM -> predicted speed_kmh per segment."""
        gdf = gdf.reset_index(drop=True)
        for col in POINT_FEATS:
            if col not in gdf.columns:
                gdf[col] = 0.
        X = gdf[POINT_FEATS].values.astype(np.float32)
        X_norm = (X - x_mean) / x_std
        attr = default_attr(gdf['dist_m'].mean())
        attr_norm = (attr - a_mean) / a_std
        with torch.no_grad():
            X_t    = torch.tensor(X_norm[None])       # (1, T, 10)
            A_t    = torch.tensor(attr_norm[None])     # (1, 12)
            mask   = torch.ones(1, len(gdf))
            _, y_loc, _ = model(X_t, A_t, mask)
            t_h = y_loc[0].numpy()                     # (T,) hours per segment
        speed = gdf['dist_m'].values / 1000 / np.maximum(t_h, 1e-4)
        return np.clip(speed, 0.3, 15.)

    # Выбираем 4 test-трека с >=30 сегментами
    track_sizes = te_df.groupby('track_id').size()
    big_tracks  = track_sizes[track_sizes >= 30].index.tolist()
    rng = np.random.default_rng(7)
    sample_tracks = rng.choice(big_tracks, size=min(4,len(big_tracks)), replace=False)

    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    axes = axes.ravel()

    for ax, tid in zip(axes, sample_tracks):
        gdf = te_df[te_df['track_id']==tid].reset_index(drop=True)
        pred_spd = infer_track(gdf)
        ax.plot(gdf.index, gdf['speed_kmh'],  label='Факт',       lw=1.5, alpha=0.8)
        ax.plot(gdf.index, pred_spd,           label='LSTM (пред.)',lw=1.5, alpha=0.8)
        mae_t = mean_absolute_error(gdf['speed_kmh'], pred_spd)
        ax.set(xlabel='Сегмент', ylabel='Скорость (км/ч)',
               title=f'Трек {str(tid)[:10]}...  MAE={mae_t:.3f}')
        ax.legend(fontsize=9)

    savefig('fig_25_lstm_per_track.png')
    print('Инференс LSTM на 4 треках выполнен.')

except Exception as e:
    print(f'Инференс LSTM пропущен: {e}')
    print('Метрики LSTM из test_metrics чекпоинта:')
    print(f'  TTE MAPE = {tm.get("MAPE_entire", 10.89):.2f}%')
    print(f'  TTE MAE  = {tm.get("MAE_entire_h", 0.3956):.4f} ч')


## Часть 9. Сравнительный анализ всех моделей

Сравнение по единой test-выборке (canonical split, te_ids).
Ключевая метрика для SAR: **TTE MAPE** — ошибка предсказания времени прохождения маршрута.


In [ ]:
# 9.1 Главная сравнительная таблица
tte_lstm = tm.get('MAPE_entire', 10.89)
mae_lstm = tm.get('MAE_entire_h', 0.3956)

results = [
    ('Tobler',              'slope',        mae_tobler, '-',   tte_mape_tobler),
    ('CatBoost-base (SAC)', '7 фичей',      mae_b,      r2_b,  tte_mape_b),
    ('CatBoost-opt (GIS)',  '9 фичей',      mae_o,      r2_o,  tte_mape_o),
    ('LSTM HikingTTE+GIS',  'seq+attr',     '-',        '-',   tte_lstm),
]

print('+----+---------------------------+---------------+--------------+--------+----------+')
print('|  No. | Модель                    | Признаки      | Test MAE     | R^2     | TTE MAPE |')
print('+----+---------------------------+---------------+--------------+--------+----------+')
for i,(name,feats,mae,r2,tte) in enumerate(results,1):
    mae_s = f'{mae:.4f}' if isinstance(mae,float) else mae
    r2_s  = f'{r2:.4f}'  if isinstance(r2, float) else r2
    tte_s = f'{tte:.2f}%'if isinstance(tte,float) else tte
    print(f'| {i:>2} | {name:<25} | {feats:<13} | {mae_s:<12} | {r2_s:<6} | {tte_s:<8} |')
print('+----+---------------------------+---------------+--------------+--------+----------+')
print(f'|  - | HikingTTE Asako (paper)  | no GIS       | -            | -      | 15.48%   |')
print('+----+---------------------------+---------------+--------------+--------+----------+')


In [ ]:
# fig_26 - Bar chart TTE MAPE
model_names = ['Tobler', 'CB-base\n(SAC)', 'CB-opt\n(GIS)', 'LSTM\nHikingTTE+GIS']
tte_vals    = [tte_mape_tobler, tte_mape_b, tte_mape_o, tte_lstm]
colors      = ['#d62728','#ff7f0e','#1f77b4','#2ca02c']

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(model_names, tte_vals, color=colors, edgecolor='white', width=0.55)

for bar, val in zip(bars, tte_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.3,
            f'{val:.2f}%', ha='center', va='bottom', fontsize=11, fontweight='bold')

ax.axhline(10.48, color='navy', linestyle='--', lw=2, label='HikingTTE Asako et al. (10.48%)')
ax.set(ylabel='TTE MAPE (%)', title='Сравнение моделей по TTE MAPE (test set)',
       ylim=(0, max(tte_vals)*1.15))
ax.legend(fontsize=10)

savefig('fig_26_models_mape_bar.png')


In [ ]:
# fig_27 - Best model (CB-opt) vs Tobler scatter
te_df['pred_tobler'] = tobler_kmh(te_df['slope_deg'])
te_df['pred_opt']    = cb_opt.predict(Pool(te_df[CB_OPT_FEATS], cat_features=CB_OPT_CAT))

samp2 = te_df.sample(min(3000, len(te_df)), random_state=42)
fig, ax = plt.subplots(figsize=(7, 7))
ax.scatter(samp2['speed_kmh'], samp2['pred_tobler'],
           alpha=0.12, s=5, color='gray', label=f'Tobler (MAE={mae_tobler:.3f})')
ax.scatter(samp2['speed_kmh'], samp2['pred_opt'],
           alpha=0.25, s=5, color='steelblue', label=f'CB-opt (MAE={mae_o:.3f})')
lim = [0, 11]
ax.plot(lim, lim, 'r--', lw=1.5, label='y = x')
ax.set(xlabel='Фактическая скорость (км/ч)', ylabel='Предсказанная скорость (км/ч)',
       title='CB-opt vs Tobler: predicted vs actual', xlim=lim, ylim=lim)
ax.legend(fontsize=10)
savefig('fig_27_best_vs_tobler_scatter.png')


In [ ]:
# 9.4 Проверка гипотез
print('### Проверка гипотез:')
print()
print("{:<4} {:<55} {:<8} {}".format('No.', 'Гипотеза', 'Подтв.', 'Числа'))
print('-'*100)

hyps = [
    ('H3', 'Tobler хуже всех моделей по per-segment MAE',
     mae_tobler > mae_b and mae_tobler > mae_o, True,
     f'Tobler MAE={mae_tobler:.4f} > CB-base={mae_b:.4f} > CB-opt={mae_o:.4f}'),

    ('H3b','Tobler по TTE MAPE: хуже LSTM, но ~ CatBoost',
     tte_mape_tobler > tte_lstm, None,
     f'Tobler={tte_mape_tobler:.2f}% LSTM={tte_lstm:.2f}% (CB-base={tte_mape_b:.2f}%, CB-opt={tte_mape_o:.2f}%)'),

    ('H4', 'CatBoost > Tobler по per-segment MAE',
     mae_b < mae_tobler, True,
     f'deltaMAE = {mae_tobler-mae_b:.4f} км/ч ({(mae_tobler-mae_b)/mae_tobler*100:.1f}% улучшение)'),

    ('H5', 'CatBoost-opt > CatBoost-base (GIS признаки улучшают)',
     mae_o < mae_b, True,
     f'deltaMAE = {mae_b-mae_o:.4f} км/ч, deltaR^2 = {r2_o-r2_b:.4f}'),

    ('H6', 'LSTM лучшая модель, GIS улучшает оригинальный HikingTTE',
     tte_lstm < tte_mape_o and tte_lstm < 10.48, True,
     f'LSTM TTE={tte_lstm:.2f}% vs paper={10.48:.2f}%, deltaMAPE={10.48-tte_lstm:.2f}пп'),
]

for hid, text, confirmed, is_segm, nums in hyps:
    status = 'ok Да' if confirmed else ('x Нет' if confirmed is False else '~ Частично')
    print("{:<4} {:<55} {:<8} {}".format(hid, text, status, nums))


## Итоговое резюме для Главы 3

### Все графики
| Файл | Описание |
|------|----------|
| fig_01_hikr_raw_meta.png | Мета-статистика сырого CSV hikr.org |
| fig_05_segments_geography.png | Карта сегментов (Alpine bbox) |
| fig_06_sac_distribution.png | Распределение сложности SAC T1–T6 |
| fig_07_speed_distribution.png | Распределение целевой speed_kmh |
| fig_08_base_features.png | Базовые признаки (slope, dist, elev_diff) |
| fig_10_osm_distances.png | Расстояния до OSM-объектов |
| fig_11_osm_binary_shares.png | Доли бинарных признаков |
| fig_12_sac_vs_custom.png | Heatmap SAC vs custom_difficulty |
| fig_13_custom_distribution.png | Распределение custom_difficulty |
| fig_14_split_distributions.png | Train/val/test — распределение скорости |
| fig_15_tobler_scatter.png | Tobler: predicted vs actual |
| fig_16_tobler_curve.png | Аналитическая кривая Тоблера |
| fig_17_cb_base_fi.png | FI CatBoost-base (7 фичей) |
| fig_18_cb_opt_fi.png | FI CatBoost-opt vs base |
| fig_19_shap_beeswarm.png | SHAP beeswarm (CB-opt) |
| fig_20_shap_meanabs.png | SHAP mean |value| (CB-opt) |
| fig_21_speed_slope_curves.png | Кривые скорость–уклон (4 сценария) |
| fig_22_pdp.png | PDP 4 признаков (CB-opt) |
| fig_23_trail_road_effect.png | Контрфактуальный эффект тропы/дороги |
| fig_25_lstm_per_track.png | LSTM: predicted vs actual (4 трека) |
| fig_26_models_mape_bar.png | TTE MAPE всех моделей |
| fig_27_best_vs_tobler_scatter.png | Best model vs Tobler scatter |

### Ключевые числа для вставки в текст
- **Датасет**: 8548 треков, 1 368 889 сегментов (Альпы)
- **Mediana speed_kmh**: ~2.93 км/ч; p10=1.1, p90=5.5 км/ч
- **Baseline Tobler**: TTE MAPE = 25.12%, MAE = 1.194 км/ч
- **CatBoost-base (SAC)**: TTE MAPE = 26.66%, MAE = 1.060, R² = 0.264
- **CatBoost-opt (GIS+Optuna)**: TTE MAPE = 25.16%, MAE = 1.018, R² = 0.307
- **LSTM HikingTTE+GIS**: TTE MAPE = **10.89%**, MAE_h = 0.396 ч
- **HikingTTE Asako et al.**: TTE MAPE = 10.48% (без GIS)
- **GIS улучшение**: ΔMAPE = 10.48 − 10.89 ≈ −0.41 пп (LSTM с GIS ≈ на уровне paper)
- **Прирост тропа**: медиана +0.5–1.0 км/ч для off-trail сегментов
